# Campus SVI — acquisition

Collects street-view imagery **metadata** for Indonesian university campuses and writes two
deliverables per campus. No imagery is downloaded, and no analysis happens here.

| Output | What it is |
|---|---|
| `data/points/{campus}_points.gpkg` | every image / panorama inside the boundary, with full metadata (layers: `mapillary`, `google`) |
| `data/cells/{campus}_cells.gpkg` | those points aggregated onto the analysis grid |

Every stage is checkpointed. If the runtime disconnects, re-run the same cell — finished work
is skipped and unfinished work resumes.

---
## 1 · Setup


In [ ]:
#@title Install (~1 min)
!pip install -q geopandas pyogrio streetlevel aiohttp nest-asyncio

import nest_asyncio; nest_asyncio.apply()
import streetlevel, aiohttp, geopandas as gpd
print('streetlevel', getattr(streetlevel, '__version__', '?'),
      '| aiohttp', aiohttp.__version__, '| geopandas', gpd.__version__)


In [ ]:
#@title Mount Drive and clone the repo
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/aditpradana36/campus-svi-availability.git'  #@param {type:'string'}
PROJECT_ROOT = '/content/drive/MyDrive/campus-svi-availability'  #@param {type:'string'}

import os, sys
REPO_DIR = '/content/campus-svi-acquisition'
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from campus_svi import config
config.set_root(PROJECT_ROOT)
for k, v in config.paths().items():
    print(f'{k:<16} {v}')


### Mapillary token

The only credential needed — `streetlevel` requires no Google key. Add `MAPILLARY_TOKEN` in the
Colab **Secrets** panel (key icon, left sidebar) rather than pasting it into a cell.


In [ ]:
#@title Load the token
import os
try:
    from google.colab import userdata
    os.environ['MAPILLARY_TOKEN'] = userdata.get('MAPILLARY_TOKEN')
except Exception as e:
    print('Secrets unavailable:', type(e).__name__)
    import getpass
    os.environ['MAPILLARY_TOKEN'] = getpass.getpass('MAPILLARY_TOKEN: ')

config.MAPILLARY_TOKEN = os.environ.get('MAPILLARY_TOKEN', '')
print('token:', 'ok' if config.MAPILLARY_TOKEN.startswith('MLY|') else 'MISSING')


In [ ]:
#@title Campuses found
from campus_svi import boundaries
found = boundaries.list_campuses()
print(f'{len(found)} campuses in {config.BOUNDARY_DIR}:')
for c in found:
    print(' ', c)


---
## 2 · Where everything stands

Run this any time. It reads the checkpoints, so it is the fastest way to see what is left.


In [ ]:
#@title Status
from campus_svi import pipeline
status = pipeline.status()
display(status)
print(f"\n{int(status['cells'].sum())}/{len(status)} campuses complete")


---
## 3 · Run one campus

Start here. Do a single campus first and check the numbers before launching the batch.

**What happens, in order:**

1. **grid** — square cells over the boundary in local UTM, plus coarse Mapillary seed boxes.
2. **mapillary** — adaptive quadtree inside each seed box. A whole-campus bbox is refused by
   the API, so refusal drives subdivision; a sampled verification split guards against silent
   truncation.
3. **google** — coverage tiles (all panoramas per tile, no dates), then one request per
   position for date, capture programme, copyright and historical captures.
4. **points** — dedup, clip to the true boundary, write the point deliverable.
5. **cells** — spatial join onto the grid, write the cell deliverable.


In [ ]:
#@title Run a single campus
CAMPUS = 'ui_main'  #@param {type:'string'}
CELL_SIZE_M = 100  #@param {type:'number'}
SEED_SIZE_M = 500  #@param {type:'number'}
ENRICH_GOOGLE = True  #@param {type:'boolean'}

result = pipeline.run_campus(
    CAMPUS.strip().lower(),
    cell_size_m=CELL_SIZE_M,
    seed_size_m=SEED_SIZE_M,
    enrich=ENRICH_GOOGLE,
)


### Sanity checks on that campus

Two numbers matter. **`depth_exhausted`** in the Mapillary output means the quadtree hit its
ceiling with boxes still being refused — raise `config.MLY_MAX_DEPTH` and re-run if you see it.
**Coverage of 0%** for a source usually means a token problem or a boundary in the wrong place,
not an absence of imagery.


In [ ]:
#@title Inspect the outputs
from campus_svi import points, cells
import pandas as pd

cid = CAMPUS.strip().lower()

for layer in ('mapillary', 'google'):
    gdf = points.load_points(cid, layer)
    print(f'--- {layer}: {len(gdf)} points ---')
    if len(gdf):
        cols = [c for c in ('year', 'capture_source', 'is_third_party',
                            'creator_username', 'camera_type') if c in gdf.columns]
        for c in cols[:3]:
            vc = gdf[c].value_counts(dropna=False).head(4).to_dict()
            print(f'  {c}: {vc}')

cdf = cells.load_cells(cid)
print(f'\n--- cells: {len(cdf)} ---')
display(cdf.drop(columns='geometry').head())


---
## 4 · Run every campus

One campus at a time by design: it keeps request rates civil and makes any failure easy to
attribute. A campus that fails does not stop the rest.

This is a long run. Everything writes to Drive as it goes, so a disconnect costs only the
campus in flight — and re-running this cell resumes it.

> Start the batch **after** a single campus has come out clean. Twenty-two campuses is a poor
> place to discover a boundary CRS problem.


In [ ]:
#@title Run all campuses
ONLY_MISSING = True  #@param {type:'boolean'}

#@markdown `ONLY_MISSING` skips campuses that already have cell data, so this
#@markdown cell is safe to re-run after an interruption.

targets = boundaries.list_campuses()
if ONLY_MISSING:
    st = pipeline.status()
    targets = st.loc[~st['cells'], 'campus_id'].tolist()

print(f'{len(targets)} campuses to run:', targets)

if targets:
    log = pipeline.run_all(targets, cell_size_m=CELL_SIZE_M,
                           seed_size_m=SEED_SIZE_M, enrich=ENRICH_GOOGLE)
    display(log)
else:
    print('nothing to do')


In [ ]:
#@title Retry any campus that failed
st = pipeline.status()
missing = st.loc[~st['cells'], 'campus_id'].tolist()
print('still missing:', missing or 'none')

#@markdown Failures are usually transient. Re-running resumes from the last
#@markdown checkpoint rather than starting the campus over.
if missing:
    log = pipeline.run_all(missing, cell_size_m=CELL_SIZE_M,
                           seed_size_m=SEED_SIZE_M, enrich=ENRICH_GOOGLE)
    display(log)


---
## 5 · Combine

Stacks every per-campus cell table into one CSV — the handoff to the analysis stage.


In [ ]:
#@title Combined cell table
st = pipeline.status()
done = st.loc[st['cells'], 'campus_id'].tolist()
combined = cells.combine(done)

if len(combined):
    print(f"\n{len(combined)} cells across {combined['campus_id'].nunique()} campuses")
    summary = combined.groupby('campus_id').agg(
        cells=('grid_id', 'count'),
        mly_cov=('mly_coverage', 'mean'),
        ggl_cov=('ggl_coverage', 'mean'),
        either=('either_coverage', 'mean'),
        mly_images=('mly_count', 'sum'),
        ggl_panos=('ggl_count', 'sum'),
    ).round(3)
    display(summary)


---
## Notes

**Tuning.** If Mapillary requests start failing in bulk, lower `config.MLY_CONCURRENCY` before
raising `config.MLY_SLEEP`. Same for `config.GOOGLE_CONCURRENCY`. The page limit ratchets down
on its own once the API refuses a size, and the settled value is reported at the end of each
campus.

**Verification.** Only `config.MLY_VERIFY_FRACTION` (default 0.1) of accepted Mapillary boxes
gets the four-quadrant truncation check, because the API refuses oversized boxes explicitly
rather than truncating silently. Set it to 1.0 for a campus you want fully verified — expensive,
but worth doing once as evidence for the methods section.

**Provenance.** `streetlevel` wraps undocumented Google endpoints that can change without
notice. Record the version and the collection date alongside the data.

**Next.** Analysis is a separate repo stage and reads `data/cells/` and `data/points/`.
Nothing here computes coverage ratios, decay curves or figures.
